# Experiment 2: conservative feature engineering

This experiment isolates feature engineering while holding the estimator, preprocessing strategy, metric, and the exact five shuffled stratified folds fixed. Feature decisions use local cross-validation only; the Kaggle leaderboard is not consulted.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data' / 'raw'
SUBMISSION_DIR = ROOT / 'submissions'

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'gender_submission.csv')
X = train.drop(columns='Survived')
y = train['Survived']
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(X, y))
print(f'Train: {train.shape}; test: {test.shape}; fixed folds: {len(folds)}')

Train: (891, 12); test: (418, 11); fixed folds: 5


## Evaluation machinery

The baseline uses median imputation plus standardization for numeric columns, and most-frequent imputation plus one-hot encoding for categorical columns. Logistic regression remains `max_iter=1000, random_state=42`. Row-wise features are deterministic. Ticket group counts are learned separately from each fold's training partition; an unseen validation ticket receives group size 1. This avoids target leakage and avoids using validation-row frequencies.

In [2]:
BASE_NUMERIC = ['Age', 'SibSp', 'Parch', 'Fare']
BASE_CATEGORICAL = ['Pclass', 'Sex', 'Embarked']
COMMON_TITLES = {'Mr', 'Miss', 'Mrs', 'Master'}

def extract_title(names):
    titles = names.str.extract(r',\s*([^.]*)\.', expand=False).str.strip()
    return titles.where(titles.isin(COMMON_TITLES), 'Rare')

def engineer_pair(fit_df, apply_df, features):
    fit_df, apply_df = fit_df.copy(), apply_df.copy()
    numeric, categorical = BASE_NUMERIC.copy(), BASE_CATEGORICAL.copy()
    if 'family_size' in features:
        for df in (fit_df, apply_df):
            df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
        numeric.append('FamilySize')
    if 'is_alone' in features:
        for df in (fit_df, apply_df):
            df['IsAlone'] = ((df['SibSp'] + df['Parch']) == 0).astype(int)
        categorical.append('IsAlone')
    if 'title' in features:
        fit_df['Title'] = extract_title(fit_df['Name'])
        apply_df['Title'] = extract_title(apply_df['Name'])
        categorical.append('Title')
    if 'cabin_known' in features:
        fit_df['CabinKnown'] = fit_df['Cabin'].notna().astype(int)
        apply_df['CabinKnown'] = apply_df['Cabin'].notna().astype(int)
        categorical.append('CabinKnown')
    if 'deck' in features:
        fit_df['Deck'] = fit_df['Cabin'].str[0].fillna('Unknown')
        apply_df['Deck'] = apply_df['Cabin'].str[0].fillna('Unknown')
        categorical.append('Deck')
    if 'ticket_group_size' in features:
        counts = fit_df['Ticket'].value_counts()
        fit_df['TicketGroupSize'] = fit_df['Ticket'].map(counts).astype(float)
        apply_df['TicketGroupSize'] = apply_df['Ticket'].map(counts).fillna(1).astype(float)
        numeric.append('TicketGroupSize')
    return fit_df, apply_df, numeric, categorical

def make_model(numeric, categorical):
    preprocessing = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical),
    ])
    return Pipeline([
        ('preprocessing', preprocessing),
        ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])

def evaluate(features=()):
    scores = []
    for train_idx, valid_idx in folds:
        fold_train, fold_valid, numeric, categorical = engineer_pair(
            X.iloc[train_idx], X.iloc[valid_idx], features
        )
        model = make_model(numeric, categorical)
        model.fit(fold_train, y.iloc[train_idx])
        scores.append(accuracy_score(y.iloc[valid_idx], model.predict(fold_valid)))
    return np.array(scores)

## Baseline reproduction

First, reproduce the unchanged baseline as a direct control. The expected reported result is 0.7969 ± 0.0146.

In [3]:
baseline_scores = evaluate()
print('Fold scores:', np.round(baseline_scores, 4).tolist())
print(f'Mean ± std: {baseline_scores.mean():.4f} ± {baseline_scores.std():.4f}')
assert np.isclose(baseline_scores.mean(), 0.7969, atol=5e-5)

Fold scores: [0.7821, 0.8034, 0.7978, 0.7809, 0.8202]
Mean ± std: 0.7969 ± 0.0146


## Individual feature hypotheses

- **FamilySize:** companions may provide help, while very large families may face coordination and access constraints.
- **IsAlone:** being unaccompanied may change access to help and evacuation behavior. It is tested separately because a threshold can express something a linear family-size term cannot.
- **Title:** a compact, interpretable extraction from `Name` may proxy age, sex, social role, and status. Rare titles are pooled to avoid sparse categories.
- **CabinKnown:** a recorded cabin may proxy passenger class, wealth, or location/access.
- **Deck:** the first cabin letter may approximate location on the ship. Missing cabins form an explicit `Unknown` category; this is a clean low-cardinality encoding.
- **TicketGroupSize:** passengers sharing a ticket may travel together and experience correlated circumstances. Counts use predictors only and are learned within each training fold.

In [4]:
experiments = {
    'Baseline': (),
    'FamilySize': ('family_size',),
    'IsAlone': ('is_alone',),
    'FamilySize + IsAlone': ('family_size', 'is_alone'),
    'Title': ('title',),
    'CabinKnown': ('cabin_known',),
    'Deck': ('deck',),
    'TicketGroupSize': ('ticket_group_size',),
}
experiment_scores = {name: evaluate(features) for name, features in experiments.items()}
rows = []
for name, scores in experiment_scores.items():
    rows.append({
        'Experiment': name,
        **{f'Fold {i}': score for i, score in enumerate(scores, 1)},
        'Mean': scores.mean(),
        'Std': scores.std(),
        'Delta vs baseline': scores.mean() - baseline_scores.mean(),
    })
results = pd.DataFrame(rows).set_index('Experiment')
display(results.style.format('{:.4f}'))

,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Mean,Std,Delta vs baseline
Experiment,,,,,,,,
Baseline,0.7821,0.8034,0.7978,0.7809,0.8202,0.7969,0.0146,0.0000
FamilySize,0.7877,0.8034,0.7978,0.7809,0.8202,0.7980,0.0136,0.0011
IsAlone,0.7765,0.7921,0.8034,0.7921,0.8315,0.7991,0.0183,0.0023
FamilySize + IsAlone,0.7765,0.7921,0.7978,0.7921,0.8315,0.7980,0.0182,0.0011
Title,0.8324,0.8090,0.8427,0.8202,0.8427,0.8294,0.0131,0.0325
CabinKnown,0.8045,0.8090,0.7697,0.7865,0.8427,0.8025,0.0245,0.0056
Deck,0.8156,0.8146,0.7640,0.7809,0.8483,0.8047,0.0295,0.0078
TicketGroupSize,0.8101,0.8034,0.7921,0.7865,0.8258,0.8036,0.0139,0.0067


## Conservative final-set selection

`Title` is the strongest individual feature. To avoid accumulating marginal features, use forward validation from that anchor. Test each individually positive candidate as an addition. A feature is retained only if it raises mean CV accuracy over the current set; no leaderboard result participates.

In [5]:
forward_candidates = {
    'Title only': ('title',),
    'Title + FamilySize': ('title', 'family_size'),
    'Title + IsAlone': ('title', 'is_alone'),
    'Title + CabinKnown': ('title', 'cabin_known'),
    'Title + Deck': ('title', 'deck'),
    'Title + TicketGroupSize': ('title', 'ticket_group_size'),
}
forward_rows = []
for name, features in forward_candidates.items():
    scores = evaluate(features)
    forward_rows.append({'Candidate': name, 'Mean': scores.mean(), 'Std': scores.std(),
                         'Delta vs baseline': scores.mean() - baseline_scores.mean()})
forward_results = pd.DataFrame(forward_rows).set_index('Candidate')
display(forward_results.style.format('{:.4f}'))

FINAL_FEATURES = ('title', 'is_alone')
final_scores = evaluate(FINAL_FEATURES)
print('Selected: baseline + Title + IsAlone')
print('Final fold scores:', np.round(final_scores, 4).tolist())
print(f'Final mean ± std: {final_scores.mean():.4f} ± {final_scores.std():.4f}')
print(f'Change vs baseline: {final_scores.mean() - baseline_scores.mean():+.4f}')

,Mean,Std,Delta vs baseline
Candidate,,,
Title only,0.8294,0.0131,0.0325
Title + FamilySize,0.8294,0.0131,0.0325
Title + IsAlone,0.8305,0.0084,0.0336
Title + CabinKnown,0.8283,0.0113,0.0314
Title + Deck,0.8249,0.0191,0.0280
Title + TicketGroupSize,0.8283,0.0104,0.0314


Selected: baseline + Title + IsAlone
Final fold scores: [0.838, 0.8146, 0.8371, 0.8315, 0.8315]
Final mean ± std: 0.8305 ± 0.0084
Change vs baseline: +0.0336


The final feature set keeps `Title` and `IsAlone`. `FamilySize` is not retained because it adds no improvement to `Title`; cabin features and ticket group size also reduce the mean from the title anchor. Thus, even individually positive features can be rejected when they do not add complementary validation signal.

## Fit all training rows and create the submission

The final model is fitted on all training data. Verification checks the exact schema, row count, passenger ordering, integer dtypes, missingness, and allowed binary predictions against the supplied test and sample-submission files.

In [6]:
full_train, engineered_test, numeric, categorical = engineer_pair(X, test, FINAL_FEATURES)
final_model = make_model(numeric, categorical)
final_model.fit(full_train, y)
predictions = final_model.predict(engineered_test).astype(int)
submission = pd.DataFrame({'PassengerId': test['PassengerId'].astype(int), 'Survived': predictions})

assert submission.columns.tolist() == sample_submission.columns.tolist() == ['PassengerId', 'Survived']
assert len(submission) == len(test) == len(sample_submission)
assert submission['PassengerId'].equals(test['PassengerId'].astype(int))
assert submission['PassengerId'].equals(sample_submission['PassengerId'].astype(int))
assert pd.api.types.is_integer_dtype(submission['PassengerId'])
assert pd.api.types.is_integer_dtype(submission['Survived'])
assert not submission.isna().any().any()
assert set(submission['Survived'].unique()).issubset({0, 1})
assert submission['PassengerId'].is_unique

SUBMISSION_DIR.mkdir(exist_ok=True)
submission_path = SUBMISSION_DIR / 'submission_02_features.csv'
submission.to_csv(submission_path, index=False)
reloaded = pd.read_csv(submission_path)
pd.testing.assert_frame_equal(reloaded, submission)
print(f'Wrote and verified {submission_path.relative_to(ROOT)} ({len(submission)} rows)')
print(submission['Survived'].value_counts().sort_index())
display(submission.head())

Wrote and verified submissions/submission_02_features.csv (418 rows)
Survived
0    253
1    165
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
